# einops-reduce-min — ex2: 2x2 min-pool a feature map via einops patch reduction

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-reduce-min`. Running the final beacon cell reports progress against the `Einops: Reduce with min` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Reduce with min` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einops-reduce-min`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-reduce-min"
DD_SUBTOPIC = "Einops: Reduce with min"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## einops.reduce — patch-pool patterns

`einops.reduce` is the toolkit's most expressive op: it can replicate ALL of `mean/max/min/sum` AND any window-pool by adding parenthesised composites on the LHS that get reduced:

```python
out = reduce(x, 'b c (h p1) (w p2) -> b c h w', 'min', p1=2, p2=2)
```

**This drill (ex2) vs ex1.** ex1 used the simple `'b c h w -> b c'` shape (collapse ALL spatial axes to a single per-channel floor). ex2 uses the much more interesting **patch-pool** shape — collapse `p1×p2` sub-blocks within `h, w` but KEEP a coarser spatial grid. Same `'min'` op, structurally richer pattern.

### Exercise 2 — 2x2 min-pool a feature map via einops patch reduction

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `einops.reduce(..., 'min')` with the patch pattern `'b c (h p1) (w p2) -> b c h w'` to perform 2x2 min-pooling on a `(B, C, H, W)` feature map, producing a `(B, C, H/2, W/2)` downsampled tensor.
> Keywords: min-pool, patch-pool, reduce, downsample
> ```

**KCs targeted:** `reduce-min-op`, `reduce-patch-pattern-via-parens`

Implement `ex2_min_pool_2x2(x)`.

Given a `(B, C, H, W)` float tensor where `H` and `W` are both **even**, perform 2x2 min-pooling — partition the spatial grid into non-overlapping `2×2` patches and take the min within each.

**Rules.**
1. Use `einops.reduce` with the patch pattern `'b c (h p1) (w p2) -> b c h w'` and `reduction='min'`.
2. Pass `p1=2, p2=2` as keyword args.
3. The output shape must be `(B, C, H//2, W//2)`. Each output cell `(h, w)` is the minimum of the four input cells `(2h..2h+1, 2w..2w+1)`.
4. Do NOT use `F.max_pool2d` with a negated input — the point is to express the pool directly via the patch pattern.

Inputs:
- `x`: `(B, C, H, W)` float tensor, `H, W` even.

Output: `(B, C, H//2, W//2)` float tensor of per-patch minima.

In [ ]:
def ex2_min_pool_2x2(x: Tensor) -> Tensor:
    """2x2 min-pool via einops patch reduction."""
    raise NotImplementedError()


def _test_ex2():
    # Hand-crafted (1, 1, 4, 4) input — the patch mins are easy to read off.
    x = t.tensor([[[[ 4.,  2.,  9.,  7.],
                    [ 1.,  3.,  8.,  6.],
                    [12., 10.,  0., -1.],
                    [11., 13., -2., -3.]]]])
    # 2x2 patches:  TL min(4,2,1,3)=1   TR min(9,7,8,6)=6
    #               BL min(12,10,11,13)=10  BR min(0,-1,-2,-3)=-3
    expected = t.tensor([[[[ 1.,  6.],
                           [10., -3.]]]])
    out = ex2_min_pool_2x2(x)
    assert out.shape == (1, 1, 2, 2), f'shape {tuple(out.shape)} != (1,1,2,2)'
    assert t.equal(out, expected), f'wrong pooled values:\n{out}\nvs\n{expected}'

    # Larger batched test — verify against a direct loop reference.
    B, C, H, W = 2, 3, 6, 8
    rng = t.Generator().manual_seed(7)
    x_big = t.randn(B, C, H, W, generator=rng)
    out_big = ex2_min_pool_2x2(x_big)
    assert out_big.shape == (B, C, H // 2, W // 2)
    for b in range(B):
        for c in range(C):
            for h in range(H // 2):
                for w in range(W // 2):
                    patch = x_big[b, c, 2*h:2*h+2, 2*w:2*w+2]
                    ref = patch.min().item()
                    got = out_big[b, c, h, w].item()
                    assert abs(got - ref) < 1e-6, f'cell ({b},{c},{h},{w}): got {got}, ref {ref}'

    # Sanity vs negated max-pool — must agree.
    import torch.nn.functional as F
    ref = -F.max_pool2d(-x_big, kernel_size=2, stride=2)
    assert t.allclose(out_big, ref, atol=1e-6), 'must equal -max_pool2d(-x)'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_min_pool_2x2(x: Tensor) -> Tensor:
    return einops.reduce(x, 'b c (h p1) (w p2) -> b c h w', 'min', p1=2, p2=2)
```

**Why this pattern works.** Each parenthesised `(h p1)` says: "the input has an `h*p1`-long axis; treat it as a grid of `h` patches of size `p1`". Any axis present on the LHS but ABSENT from the RHS gets reduced. So `p1` and `p2` are reduced (collapsed into the specified `reduction='min'`), while `h` and `w` survive as the coarser output spatial axes.

**Why `'min'` is unusual.** PyTorch ships `F.max_pool2d` and `F.avg_pool2d` but no `min_pool2d` — the standard trick is `-max_pool2d(-x)`. einops's `reduce` makes the direct expression possible *without* sign-flipping, which matters when `x` has been passed through a layer that's sensitive to the sign (e.g. an intermediate that's clamped to non-negative).

**Difference from ex1.** ex1 did the *total* spatial floor `'b c h w -> b c'` — collapse everything to a single value per channel. ex2 keeps a coarser spatial grid intact, which is what real min-pool / max-pool layers do.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()